In [1]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
from scipy import stats
from IPython.display import display

In [2]:
TASK = Path.cwd().parent
ROOT = TASK.parent
INPUT, OUTPUT = TASK/'input', TASK/'output'
CONFIG = json.loads((ROOT/'config.json').read_text())
EPS, Y, SEED = CONFIG['eps'], CONFIG['last_year'], CONFIG['seed']

In [3]:
panel = pd.read_csv(INPUT/'fit/data/panel.csv',index_col='dblp_id')
Q = panel.to_numpy(dtype=float).T
complete = Q[:,np.isfinite(Q).all(axis=0)]
models = {tag:np.load(INPUT/tag/'trajectories.npy') for tag in CONFIG['models']}
# complete empirical careers for cross-career comparisons; available follow-up exported separately
sources = {'empirical_complete':complete, **models}
available = {'empirical_available':Q, **sources}
STAGES = [('pooled',0,Y-1), ('0',0,0), ('1-4',1,4), ('5-7',5,7), ('8-20',8,Y-1)]
WINDOWS = {'stage_0':(0,0), 'stage_1_4':(1,4), 'stage_5_7':(5,7), 'stage_8_20':(8,Y),
           'through_5':(0,5), 'last_four':(Y-3,Y), 'career_total':(0,Y), 'generated_common':(6,Y)}

In [4]:
def moments(values):
    q = np.asarray(values)
    q = q[np.isfinite(q)]
    variable = len(q)>3 and np.ptp(q)>0
    return {'n':len(q), 'mean':q.mean(), 'median':np.median(q), 'variance':q.var(ddof=0), 'sd':q.std(ddof=0),
            'skewness':stats.skew(q,bias=False) if variable else np.nan,
            'excess_kurtosis':stats.kurtosis(q,bias=False) if variable else np.nan,
            'q05':np.quantile(q,.05), 'q95':np.quantile(q,.95)}
def cdf_knots(values):
    q = np.sort(np.asarray(values)[np.isfinite(values)])
    # sampled knots of the exact ECDF; retain both sides of each jump and zero mass
    x = np.unique(np.r_[q[0],np.quantile(q,np.linspace(0,1,401)),0.,q[-1]])
    return x,np.searchsorted(q,x,side='left')/len(q),np.searchsorted(q,x,side='right')/len(q)
def shape_fit(values,family):
    q = np.asarray(values)[np.isfinite(values)]
    dist = getattr(stats,family)
    loc,scale = dist.fit(q)
    if scale<=0:
        return {'n':len(q),'loc':loc,'fit_scale':scale,'qq_r':np.nan,'qq_r2':np.nan,'ks_D':np.nan}
    r = stats.probplot(q,dist=family,fit=True)[1][2]
    return {'n':len(q),'loc':loc,'fit_scale':scale,'qq_r':r,'qq_r2':r*r,
            'ks_D':stats.kstest(q,family,args=(loc,scale)).statistic}
def histogram(values,family):
    q = np.asarray(values)[np.isfinite(values)]
    lo,hi = np.quantile(q,[.005,.995])
    edges = np.linspace(lo,hi,61)
    count,_ = np.histogram(q,bins=edges)
    center = (edges[:-1]+edges[1:])/2
    fit = shape_fit(q,family)
    pdf = getattr(stats,family).pdf(center,loc=fit['loc'],scale=fit['fit_scale']) if fit['fit_scale']>0 else np.full(len(center),np.nan)
    return pd.DataFrame({'x':center,'width':np.diff(edges),'density':count/(len(q)*np.diff(edges)),
                         'fit_density':pdf,'display_mass':count.sum()/len(q)})

In [5]:
rows = []
for source,array in available.items():
    for t in range(Y+1):
        q = array[t,np.isfinite(array[t])]
        for scale,values in [('raw',q),('offset_log',np.log(q+EPS))]:
            rows.append({'source':source,'age':t,'scale':scale,'empirical_prefix':source=='ar6_6' and t<6,**moments(values)})
count_moments = pd.DataFrame(rows)
count_moments.to_csv(OUTPUT/'count_moments.csv',index=False)
display(count_moments.loc[count_moments.age.eq(Y)].head(8))

,source,age,scale,empirical_prefix,n,mean,median,variance,sd,skewness,excess_kurtosis,q05,q95
40,empirical_available,20,raw,False,591,6.547803,4.493036,5.238966e+01,7.238070,2.040154,5.587890,0.000000,21.246465
41,empirical_available,20,offset_log,False,591,1.390287,1.606039,1.387430e+00,1.177892,-0.438407,-0.632737,-0.713350,3.078930
82,empirical_complete,20,raw,False,510,6.789832,4.649225,5.376726e+01,7.332616,2.007214,5.391408,0.000000,21.640393
83,empirical_complete,20,offset_log,False,510,1.447582,1.636896,1.331617e+00,1.153957,-0.480611,-0.503176,-0.713350,3.096932
124,arw4,20,raw,False,10000,6.666496,5.596899,2.566399e+01,5.065964,1.293687,2.429601,0.682687,16.379233
125,arw4,20,offset_log,False,10000,1.691646,1.806139,6.469145e-01,0.804310,-0.574035,-0.033054,0.159298,2.825491
166,lognormal_grw,20,raw,False,10000,17713.294556,10.593780,1.525501e+11,390576.585219,42.202975,2071.400579,0.000000,6861.429471
167,lognormal_grw,20,offset_log,False,10000,2.966777,2.405483,9.380281e+00,3.062724,0.914048,0.545882,-0.713350,8.833742


In [6]:
moment_rows,fit_rows,hist_frames,cdf_frames = [],[],[],[]
for source,array in available.items():
    raw = array[1:]-array[:-1]
    logged = np.log(array[1:]+EPS)-np.log(array[:-1]+EPS)
    for scale,delta,family in [('raw',raw,'laplace'),('offset_log',logged,'norm')]:
        for t in range(Y):
            moment_rows.append({'source':source,'age':t,'scale':scale,
                                'empirical_prefix':source=='ar6_6' and t<5,**moments(delta[t])})
        for stage,start,end in STAGES:
            values = delta[start:end+1].ravel()
            values = values[np.isfinite(values)]
            fit_rows.append({'source':source,'stage':stage,'scale':scale,'family':family,'start':start,'end':end,
                             'prefix_included':source=='ar6_6' and start<6,**shape_fit(values,family)})
            hist_frames.append(histogram(values,family).assign(source=source,stage=stage,scale=scale))
            x,left,right = cdf_knots(values)
            cdf_frames.append(pd.DataFrame({'x':x,'cdf_left':left,'cdf':right,'n':len(values),
                                            'source':source,'stage':stage,'scale':scale}))
pd.DataFrame(moment_rows).to_csv(OUTPUT/'increment_moments.csv',index=False)
pd.DataFrame(fit_rows).to_csv(OUTPUT/'increment_fits.csv',index=False)
pd.concat(hist_frames,ignore_index=True).to_csv(OUTPUT/'increment_histograms.csv',index=False)
pd.concat(cdf_frames,ignore_index=True).to_csv(OUTPUT/'increment_cdf.csv',index=False)

In [7]:
# windowed totals and career diagnostics; no scholar-level records exported
samples = {}
for source,array in sources.items():
    samples[source] = {window:array[start:end+1].sum(axis=0) for window,(start,end) in WINDOWS.items()}
    samples[source].update({'annual_pooled':array.ravel(), 'career_sd':array.std(axis=0),
                            'career_sd_positive':np.array([v[v>0].std() if (v>0).sum()>1 else np.nan for v in array.T]),
                            'zero_count':np.sum(array==0,axis=0),'year_max':array.argmax(axis=0)})
    for t in [0,4,7,13,Y]: samples[source][f'annual_{t}'] = array[t]
# historical initialization used all observed q0 rather than the survivor cohort
samples['empirical_initial'] = {'initial':Q[0,np.isfinite(Q[0])]}
for source,array in models.items():samples[source]['initial'] = array[0]
# historical last-four observed careers, explicitly distinct from common years 17–20
last = []
for career in Q.T:
    observed = np.flatnonzero(np.isfinite(career))
    if len(observed)>=4 and np.all(np.diff(observed[-4:])==1):last.append(career[observed[-4:]].sum())
samples['empirical_last_observed'] = {'last_four':np.asarray(last)}
cdfs,summaries,comparisons = [],[],[]
for source,diagnostics in samples.items():
    for diagnostic,values in diagnostics.items():
        values = values[np.isfinite(values)]
        x,left,right = cdf_knots(values)
        cdfs.append(pd.DataFrame({'source':source,'diagnostic':diagnostic,'x':x,'cdf_left':left,'cdf':right,'n':len(values)}))
        summaries.append({'source':source,'diagnostic':diagnostic,**moments(values)})
for tag in CONFIG['models']:
    for diagnostic in samples['empirical_complete']:
        obs = samples['empirical_complete'][diagnostic]
        sim = samples[tag][diagnostic]
        obs,sim = obs[np.isfinite(obs)],sim[np.isfinite(sim)]
        ks = stats.ks_2samp(obs,sim,method='asymp').statistic
        comparisons.append({'source':tag,'diagnostic':diagnostic,'n_empirical':len(obs),'n_simulated':len(sim),
                            'ks_D':ks,'wasserstein':stats.wasserstein_distance(obs,sim),
                            'prefix_included':tag=='ar6_6' and diagnostic not in ['last_four','generated_common','stage_8_20','annual_7','annual_13',f'annual_{Y}']})
pd.concat(cdfs,ignore_index=True).to_csv(OUTPUT/'distribution_cdf.csv',index=False)
pd.DataFrame(summaries).to_csv(OUTPUT/'historic_distribution_summary.csv',index=False)
pd.DataFrame(comparisons).to_csv(OUTPUT/'distribution_distances.csv',index=False)

In [8]:
qq_frames,qq_summaries,log_hist = [],[],[]
for source,diagnostics in samples.items():
    for window in WINDOWS:
        if window not in diagnostics:continue
        q = diagnostics[window]
        logq = np.log2(q[q>0])
        fit = shape_fit(logq,'norm')
        qq_summaries.append({'source':source,'window':window,'n_total':len(q),'n_positive':len(logq),'prefix_included':source=='ar6_6' and WINDOWS[window][0]<6,**fit})
        theoretical,ordered = stats.probplot((logq-logq.mean())/logq.std(ddof=1),dist='norm',fit=False)
        # every ordered value for QQ R; compact quantile knots for the plotted curve
        keep = np.unique(np.linspace(0,len(ordered)-1,min(401,len(ordered))).astype(int))
        qq_frames.append(pd.DataFrame({'source':source,'window':window,'normal_quantile':theoretical[keep],
                                       'observed_quantile':ordered[keep]}))
        log_hist.append(histogram(logq,'norm').assign(source=source,window=window))
pd.DataFrame(qq_summaries).to_csv(OUTPUT/'window_qq_summary.csv',index=False)
pd.concat(qq_frames,ignore_index=True).to_csv(OUTPUT/'window_qq_points.csv',index=False)
pd.concat(log_hist,ignore_index=True).to_csv(OUTPUT/'total_log2_histograms.csv',index=False)

In [9]:
# rounded count PMF with an explicit overflow bin; no silent clipping into count 20
pmf_rows,survival_rows = [],[]
for source,array in sources.items():
    rounded = np.rint(array.ravel())
    for k in range(21):
        probability = np.mean(rounded==k) if k<20 else np.mean(rounded>=20)
        pmf_rows.append({'source':source,'count':k,'label':str(k) if k<20 else '20+', 'probability':probability})
    zeros = np.sum(array==0,axis=0)
    for k in range(Y+2):survival_rows.append({'source':source,'zero_years':k,'survival':np.mean(zeros>=k),'n':len(zeros)})
pd.DataFrame(pmf_rows).to_csv(OUTPUT/'annual_rounded_pmf.csv',index=False)
pd.DataFrame(survival_rows).to_csv(OUTPUT/'zero_year_survival.csv',index=False)

In [10]:
# fixed empirical bin edges, shared across models; starts 0, 1–4, 5–7, 8–19
rows = []
for stage,start,end in STAGES:
    current,nxt = Q[start:end+1].ravel(),Q[start+1:end+2].ravel()
    obs = current[np.isfinite(current)&np.isfinite(nxt)&(current>0)]
    edges = np.unique(np.quantile(obs,np.linspace(0,1,11)))
    edges[0],edges[-1] = -np.inf,np.inf
    for source,array in {'empirical_available':Q,**models}.items():
        current,nxt = array[start:end+1].ravel(),array[start+1:end+2].ravel()
        valid = np.isfinite(current)&np.isfinite(nxt)&(current>0)
        frame = pd.DataFrame({'q':current[valid],'dropout':nxt[valid]==0})
        frame['bin'] = pd.cut(frame.q,edges,labels=False,include_lowest=True)
        grouped = frame.groupby('bin').agg(n=('q','size'),q_mean=('q','mean'),n_dropout=('dropout','sum'),probability=('dropout','mean')).reset_index()
        grouped['source'],grouped['stage'] = source,stage
        rows.append(grouped)
pd.concat(rows,ignore_index=True).to_csv(OUTPUT/'dropout_by_productivity.csv',index=False)

In [11]:
# conditional memory at q_t near 0, 5, 10; record selected q range and any widened neighborhood
rows = []
for source,array in {'empirical_available':Q,**models}.items():
    prev,now,nxt = array[:-2].ravel(),array[1:-1].ravel(),array[2:].ravel()
    valid = np.isfinite(prev)&np.isfinite(now)&np.isfinite(nxt)
    frame = pd.DataFrame({'previous':prev[valid],'current':now[valid],'next':nxt[valid]})
    for target in [0,5,10]:
        selected = frame.loc[(frame.current-target).abs().le(.5)].copy()
        widened = len(selected)<400
        if widened:selected = frame.iloc[np.argsort((frame.current-target).abs().to_numpy(),kind='stable')[:min(2500,len(frame))]].copy()
        n_bins = min(10,selected.previous.nunique())
        if n_bins<2:continue
        selected['bin'] = pd.qcut(selected.previous,q=n_bins,labels=False,duplicates='drop')+1
        grouped = selected.groupby('bin').agg(n=('next','size'),previous_mean=('previous','mean'),
                                             next_mean=('next','mean'),next_median=('next','median')).reset_index()
        grouped['source'],grouped['target'],grouped['widened'] = source,target,widened
        grouped['current_min'],grouped['current_max'] = selected.current.min(),selected.current.max()
        rows.append(grouped)
pd.concat(rows,ignore_index=True).to_csv(OUTPUT/'conditional_memory.csv',index=False)
# annual ranks, fixed empirical survivor cohort; zero ties retain average rank
ranks = pd.read_csv(OUTPUT/'rank_correlations.csv')
ranks = ranks.loc[ranks.measure.eq('annual') & ranks.reference.isin(['age_0','lag_3','lag_5'])].copy()
ranks['reference_age'] = np.select([ranks.reference.eq('age_0'),ranks.reference.eq('lag_3')],[0,ranks.age-3],default=ranks.age-5)
ranks['both_generated'] = ~ranks.source.str.startswith('empirical') & ((ranks.source!='ar6_6') | (ranks.reference_age>=6))
ranks.to_csv(OUTPUT/'rank_mixing.csv',index=False)
display(ranks.loc[ranks.age.eq(Y),['source','reference','rho','n','both_generated']])

,source,reference,rho,n,both_generated
20,empirical_complete,age_0,0.187294,510,False
74,empirical_complete,lag_3,0.653156,510,False
90,empirical_complete,lag_5,0.582696,510,False
202,arw4,age_0,0.001876,10000,True
256,arw4,lag_3,0.202827,10000,True
272,arw4,lag_5,0.077234,10000,True
384,lognormal_grw,age_0,0.126381,10000,True
438,lognormal_grw,lag_3,0.827153,10000,True
454,lognormal_grw,lag_5,0.730615,10000,True
566,ar1_s,age_0,0.001461,10000,True
